# SI7016 - Workshop 3: RAG prototype (Colab)
Fine-tuning vs RAG on the CRP robot manuals. This notebook builds the **retrieval** side: chunking -> BM25 / dense (FAISS) / hybrid (RRF) / reranker, evaluated with Recall@K and MRR.

**Runtime:** CPU is enough for this prototype (a T4 only speeds up embeddings).  
**Inputs:** `corpus_pages.jsonl`, `eval_seed.jsonl`.  
**Note:** for the Welding manual `page` = chapter number, so its Recall@K is coarser than for the other manuals (true page numbers).

In [ ]:
!pip -q install langchain-text-splitters sentence-transformers faiss-cpu rank_bm25 pandas

In [ ]:
# STEP 1 - upload corpus_pages.jsonl and eval_seed.jsonl (or copy them to the Colab file panel)
import os
need = ["corpus_pages.jsonl", "eval_seed.jsonl"]
if not all(os.path.exists(f) for f in need):
    from google.colab import files
    print("Select both files in the dialog...")
    files.upload()
for f in need:
    print(f"{f:22s} ->", "OK, %d bytes" % os.path.getsize(f) if os.path.exists(f) else "MISSING")

In [ ]:
# STEP 2 - load corpus and evaluation set
import json, re, numpy as np, pandas as pd
from collections import defaultdict, Counter

def load_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(l) for l in f if l.strip()]

pages = load_jsonl("corpus_pages.jsonl")
evalset = load_jsonl("eval_seed.jsonl")
print("corpus records :", len(pages))
print("eval questions :", len(evalset))
print("chars per manual:")
for m, c in Counter({p["manual"]: 0 for p in pages}).items():
    print(f"  {m[:55]:55s}", sum(len(p["text"]) for p in pages if p["manual"] == m))

In [ ]:
# STEP 3 - chunking (character-based recursive splitter, keeps manual/page metadata)
from langchain_text_splitters import RecursiveCharacterTextSplitter

CHUNK_SIZE, CHUNK_OVERLAP = 1000, 150          # <- knobs to experiment with
splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP,
    separators=["\n\n", "\n", " ", ""])

chunks = []
for r in pages:
    for t in splitter.split_text(r["text"]):
        chunks.append({"chunk_id": len(chunks), "manual": r["manual"], "page": r["page"], "text": t})

lens = [len(c["text"]) for c in chunks]
print(f"chunks: {len(chunks)} | length min/median/max = {min(lens)}/{int(np.median(lens))}/{max(lens)}")
print("chunks per manual:", dict(Counter(c["manual"][:25] for c in chunks)))
print("\nexample chunk:\n", chunks[len(chunks)//2]["text"][:400])

In [ ]:
# STEP 4 - sparse retrieval (BM25). The tokenizer keeps tokens such as "M190", "0.1mm", "mm/s"
from rank_bm25 import BM25Okapi

def tokenize(text):
    return re.findall(r"[a-z0-9_]+(?:[./-][a-z0-9]+)*", text.lower())

bm25 = BM25Okapi([tokenize(c["text"]) for c in chunks])

def bm25_search(q, k=10):
    scores = bm25.get_scores(tokenize(q))
    return np.argsort(-scores)[:k].tolist()

print("BM25 ready. Test query -> top chunk pages:")
for cid in bm25_search("which relay detects arc start success", 3):
    print("  ", chunks[cid]["manual"][:30], "p.", chunks[cid]["page"])

In [ ]:
# STEP 5 - dense retrieval: sentence-transformers embeddings + FAISS (cosine via inner product)
import faiss, time
from sentence_transformers import SentenceTransformer

EMB_MODEL = "BAAI/bge-small-en-v1.5"           # <- try also "BAAI/bge-base-en-v1.5"
QUERY_PREFIX = "Represent this sentence for searching relevant passages: "
embedder = SentenceTransformer(EMB_MODEL)

t0 = time.time()
emb = embedder.encode([c["text"] for c in chunks], batch_size=64,
                      normalize_embeddings=True, show_progress_bar=True).astype("float32")
print(f"embeddings {emb.shape} in {time.time()-t0:.1f}s")

index = faiss.IndexFlatIP(emb.shape[1])
index.add(emb)
print("FAISS vectors:", index.ntotal)

def dense_search(q, k=10):
    qv = embedder.encode([QUERY_PREFIX + q], normalize_embeddings=True).astype("float32")
    _, ids = index.search(qv, k)
    return ids[0].tolist()

In [ ]:
# STEP 6 - hybrid retrieval with Reciprocal Rank Fusion (RRF)
def rrf(rank_lists, k=60, top=50):
    score = defaultdict(float)
    for rl in rank_lists:
        for rank, cid in enumerate(rl):
            score[cid] += 1.0 / (k + rank + 1)
    return [c for c, _ in sorted(score.items(), key=lambda x: -x[1])][:top]

def hybrid_search(q, k=10):
    return rrf([dense_search(q, 50), bm25_search(q, 50)])[:k]

print("hybrid ready:", [chunks[c]["page"] for c in hybrid_search("collision detection relay", 5)])

In [ ]:
# STEP 7 (optional) - cross-encoder reranker on the hybrid candidate pool
from sentence_transformers import CrossEncoder
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

def rerank_search(q, k=10, pool=30):
    cand = hybrid_search(q, pool)
    scores = reranker.predict([(q, chunks[c]["text"]) for c in cand])
    return [cand[i] for i in np.argsort(-scores)][:k]

print("reranker ready")

## Evaluation
A retrieved chunk counts as a hit if its (manual, page) is in the gold list of the question.

In [ ]:
# STEP 8 - evaluation: Recall@K and MRR at (manual, page) level
KS = (1, 3, 5, 10)

def evaluate(search_fn, ds, ks=KS):
    hits = {k: 0 for k in ks}; mrr = 0.0; rows = []
    for ex in ds:
        gold = {(x["manual"], x["page"]) for x in ex["gold"]}
        ids = search_fn(ex["question"], max(ks))
        got = [(chunks[i]["manual"], chunks[i]["page"]) for i in ids]
        first = next((r for r, p in enumerate(got, 1) if p in gold), None)
        for k in ks:
            hits[k] += int(first is not None and first <= k)
        mrr += 1.0 / first if first else 0.0
        rows.append({"id": ex["id"], "first_hit_rank": first, "top3": [p for _, p in got[:3]], "gold": sorted(p for _, p in gold)})
    n = len(ds)
    res = {f"Recall@{k}": hits[k] / n for k in ks}; res["MRR"] = mrr / n
    return res, rows

systems = {"BM25": bm25_search}
if "dense_search" in globals():  systems["Dense"] = dense_search
if "hybrid_search" in globals(): systems["Hybrid (RRF)"] = hybrid_search
if "rerank_search" in globals(): systems["Hybrid + rerank"] = rerank_search

table, details = {}, {}
for name, fn in systems.items():
    table[name], details[name] = evaluate(fn, evalset)
print(pd.DataFrame(table).T.round(3).to_string())

In [ ]:
# STEP 9 - error analysis: questions the best system missed within top-5
best = list(systems)[-1]
print("system inspected:", best)
for ex, row in zip(evalset, details[best]):
    if row["first_hit_rank"] is None or row["first_hit_rank"] > 5:
        print(f"\n[{ex['id']}] {ex['question']}")
        print("   gold chapters/pages:", row["gold"], "| retrieved top-3 pages:", row["top3"], "| first hit rank:", row["first_hit_rank"])

In [ ]:
# STEP 10 - persist chunks for the next phase (fine-tuning dataset / deployment)
with open("chunks.jsonl", "w", encoding="utf-8") as f:
    for c in chunks:
        f.write(json.dumps(c, ensure_ascii=False) + "\n")
print("saved chunks.jsonl:", len(chunks))
# faiss.write_index(index, "faiss.index")   # uncomment once the dense step has run